# 11 — Advanced OOP: Python Data Model, Protocols, and Metaprogramming

Goal: understand dunder methods, descriptors, properties, `__slots__`, ABCs, and a taste of metaclasses.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: The Python data model (dunder methods)

Python’s “magic” comes from special methods:

- `__len__`, `__iter__`, `__getitem__`
- `__repr__`, `__str__`
- `__eq__`, `__hash__`
- `__enter__`/`__exit__` (context managers)
- numeric ops: `__add__`, `__mul__`, ...

You don’t memorize all of them—you learn the *patterns*.

## 2.
L2: Implementing a small container type

We’ll implement a read-only “bag” that behaves like a collection.

In [ ]:

from collections.abc import Iterable, Iterator

class Bag:
    def __init__(self, items: Iterable[str]):
        self._items = tuple(items)

    def __repr__(self) -> str:
        return f"Bag(items={self._items!r})"

    def __len__(self) -> int:
        return len(self._items)

    def __iter__(self) -> Iterator[str]:
        return iter(self._items)

    def __contains__(self, x: object) -> bool:
        return x in self._items

b = Bag(["a","b","a"])
print(b)
print("len:", len(b))
print("a" in b, "c" in b)
print(list(b))


## 3.
L3: Equality and hashing

Rule:
- If an object is **mutable**, it usually should NOT be hashable.
- If an object is used as dict key or set element, its hash must be stable.

In [ ]:

from dataclasses import dataclass

@dataclass(frozen=True)
class Key:
    a: int
    b: int

k1 = Key(1,2)
k2 = Key(1,2)
print(k1 == k2)
s = {k1, k2}
print("set size:", len(s))


## 4.
L4: Properties and the descriptor protocol

`@property` is the friendly face of **descriptors** (objects controlling attribute access).

In [ ]:

class Circle:
    def __init__(self, r: float):
        self.r = r

    @property
    def area(self) -> float:
        import math
        return math.pi * self.r * self.r

c = Circle(2)
print(c.area)


## 5.
L5: `__slots__` (memory/performance tradeoffs)

`__slots__` can:
- reduce per-instance memory
- prevent accidental attribute creation

But it complicates multiple inheritance and some tooling.

In [ ]:

class Slotted:
    __slots__ = ("x", "y")
    def __init__(self, x: int, y: int):
        self.x = x
        self.y = y

s = Slotted(1, 2)
print(s.x, s.y)
# s.z = 3  # would raise AttributeError


## 6.
L6: ABCs and Protocols (duck typing, formalized)

- ABCs (`abc.ABC`) define *explicit* interfaces.
- Protocols (`typing.Protocol`) define *structural* interfaces (if it walks like a duck...).

In [ ]:

from abc import ABC, abstractmethod

class Serializer(ABC):
    @abstractmethod
    def dumps(self, obj) -> str: ...

class JsonSerializer(Serializer):
    def dumps(self, obj) -> str:
        import json
        return json.dumps(obj)

print(JsonSerializer().dumps({"a": 1}))


## 7.
L7: Metaclasses (very advanced; use rarely)

Metaclasses customize **class creation**.
Most problems can be solved with decorators or base classes.
This section is just enough to recognize what you’re looking at.

In [ ]:

class UpperAttrMeta(type):
    def __new__(mcls, name, bases, ns):
        ns2 = {}
        for k, v in ns.items():
            ns2[k.upper() if not k.startswith("__") else k] = v
        return super().__new__(mcls, name, bases, ns2)

class Demo(metaclass=UpperAttrMeta):
    x = 1
    def hello(self): return "hi"

d = Demo()
print(hasattr(Demo, "x"), hasattr(Demo, "X"))
print(d.HELLO())


## 8.
L8: Exercises

1. Add `__getitem__` to `Bag` to support indexing.
2. Create a `@property` that validates input via a setter.
3. Explain why a mutable object should rarely be hashable.

## 9.
L9: `__getattr__` and `__getattribute__` (attribute access hooks)

- `__getattr__` is called when normal lookup fails.
- `__getattribute__` intercepts *all* attribute access (use very carefully).

In [ ]:

class Lazy:
    def __init__(self):
        self._cache = {}

    def __getattr__(self, name: str):
        # called only if attribute not found
        if name == "expensive":
            v = 42
            self._cache[name] = v
            return v
        raise AttributeError(name)

x = Lazy()
print(x.expensive)


## 10.
L10: Descriptors (manual example)

A descriptor implements `__get__`, `__set__`, and/or `__delete__`.
Properties are descriptors.

In [ ]:

class Positive:
    def __set_name__(self, owner, name):
        self.private_name = "_" + name

    def __get__(self, obj, objtype=None):
        return getattr(obj, self.private_name)

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError("must be positive")
        setattr(obj, self.private_name, value)

class Item:
    price = Positive()
    def __init__(self, price: float):
        self.price = price

it = Item(10)
print(it.price)
